In [ ]:
# oxford pet dataset
import kagglehub

# Core
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms

# Metric Learning
from pytorch_metric_learning import losses, miners, samplers


# Data & Computation
import numpy as np
import os, re, random, itertools
from PIL import Image
from sklearn.model_selection import train_test_split

# Retrieval
from sklearn.neighbors import NearestNeighbors

# Evaluation
from sklearn.metrics import roc_curve, roc_auc_score, silhouette_score
from sklearn.manifold import TSNE  # if using t-SNE
import umap                        # if using UMAP

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Bonus (optional)
from torchcam.methods import GradCAM
from torchcam.utils import overlay_mask
from torchvision.transforms.functional import to_pil_image
import pandas as pd

# Data Preparation

In [ ]:
# Download dataset from kaggle, function also returns path to the downloaded images
imgdir = kagglehub.dataset_download("tanlikesmath/the-oxfordiiit-pet-dataset", output_dir="./data") + "/images/"
print(imgdir)

In [ ]:
def get_breed(filename):
    """
    Extracts the species and breed name from the filename."""
    name = filename.rsplit('.', 1)[0]
    breed = re.sub(r'_\d+$', '', name)
    if breed[0].isupper():
        species = 'cat'
    else:        
        species = 'dog'
    return species, breed

def build_class_to_idx(imgdir):
    """
    Builds a mapping from breed names to class indices."""
    breeds = set()
    for filename in os.listdir(imgdir):
        if filename.endswith('.jpg'):
            _ , breed = get_breed(filename)
            breeds.add(breed)
    breed_list = sorted(list(breeds))
    class_to_idx = {breed: idx for idx, breed in enumerate(breed_list)}
    species_to_idx = {'cat': 0, 'dog': 1}
    return class_to_idx, species_to_idx

In [ ]:
class PetDataset(Dataset):
    def __init__(self, imgdir, classes, speciesdict, transform=None):
        self.imgdir = imgdir
        self.classes = classes
        self.species = speciesdict
        self.transform = transform
        self.samples = []
        self.breeddict = {breed: [] for breed in self.classes.keys()}  # Initialize breed lists for each breed
        sample_idx = 0
        for filename in os.listdir(imgdir):
            if filename.endswith('.jpg'):
                species, breed = get_breed(filename)
                label = self.classes[breed]
                species_label = self.species[species]
                self.samples.append((filename, label, species_label))
                self.breeddict[breed].append(sample_idx)  # ← now correctly indexes into self.samples
                sample_idx += 1

    def __len__(self):
        return len(self.samples)
    
    def apply_transform(self, transform):
        self.transform = transform
        
    

    def __getitem__(self, idx):
        filename, label, species = self.samples[idx]
        img_path = os.path.join(self.imgdir, filename)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label, species
    
class TransformSubset(Dataset):
    """Small wrapper making it able to apply transforms on subsets of the data.
    This way we can apply different transforms to train and test data without having to duplicate the dataset.
    """
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label, species = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label, species

In [ ]:
# class to index mapping and inverse mapping
classes, speciesdict = build_class_to_idx(imgdir)
classes_inv = {idx: breed for breed, idx in classes.items()}

cat_breeds = [breed for breed in classes.keys() if breed[0].isupper()]
dog_breeds = [breed for breed in classes.keys() if breed[0].islower()]

# construct pyTorch dataset
data = PetDataset(imgdir, classes, speciesdict)

## Inspecting samples of every breed

In [ ]:
def show_species(dataset, breeds):
    num_breeds = len(breeds)
    cols, rows = 4, (num_breeds + 3) // 4
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
    for breed in breeds:
        idx = random.choice(dataset.breeddict[breed])
        image, label, _ = dataset[idx]
        ax = axes.flatten()[breeds.index(breed)]
        ax.imshow(image)
        ax.set_title(breed)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
show_species(data, cat_breeds)

In [ ]:
show_species(data, dog_breeds)

## Holding out of breeds for few-shot evaluation
5 breeds will be held out from the dataset for few-shot classification later. 
2 breeds will be similar to each other

**Held-out breeds:**
- American bulldog (dog, similar to American pit bull terrier)
- American it bull terrier (doc, similar to American bulldog)
- Saint Bernard (dog)
- Bengal (cat)
- Siamese (cat)


In [ ]:
held_out = ['american_bulldog', 'american_pit_bull_terrier', 'saint_bernard', 'Bengal', 'Siamese']
held_out_ids = [
    idx
    for breed in held_out
    for idx in data.breeddict[breed]
]

held_out_class_ids = [classes[breed] for breed in held_out]


## Creating subsets

In [ ]:
held_out_set   = torch.utils.data.Subset(data, held_out_ids)
remaining_set  = torch.utils.data.Subset(data, [idx for idx in range(len(data)) if idx not in held_out_ids])

remaining_indices = remaining_set.indices
remaining_labels  = [data.samples[idx][1] for idx in remaining_indices]

# Split indices, not the dataset
train_indices, val_indices = train_test_split(
    remaining_indices, test_size=0.2, stratify=remaining_labels, random_state=42
)

val_labels = [data.samples[idx][1] for idx in val_indices]
val_indices, test_indices = train_test_split(
    val_indices, test_size=0.5, stratify=val_labels, random_state=42
)

# Wrap back into Subsets
train_set = torch.utils.data.Subset(data, train_indices)
val_set   = torch.utils.data.Subset(data, val_indices)
test_set  = torch.utils.data.Subset(data, test_indices)

In [ ]:
# plot class distribution in train, val, test
def plot_class_distribution(train, val, test, title):
    # get labels from subsets
    trainlabels = [data.samples[idx][1] for idx in train.indices]
    vallabels = [data.samples[idx][1] for idx in val.indices]
    testlabels = [data.samples[idx][1] for idx in test.indices]
    breed_counts = {breed: 0 for breed in classes.keys()}
    for label in trainlabels:
        breed = classes_inv[label]
        breed_counts[breed] += 1
    breed_counts_val = {breed: 0 for breed in classes.keys()}
    for label in vallabels:
        breed = classes_inv[label]
        breed_counts_val[breed] += 1
    breed_counts_test = {breed: 0 for breed in classes.keys()}
    for label in testlabels:
        breed = classes_inv[label]
        breed_counts_test[breed] += 1
    breeds = list(breed_counts.keys())
    counts_train = [breed_counts[breed] for breed in breeds]
    counts_val = [breed_counts_val[breed] for breed in breeds]
    counts_test = [breed_counts_test[breed] for breed in breeds]
    x = np.arange(len(breeds))
    width = 0.25
    plt.figure(figsize=(15, 5))
    plt.bar(x - width, counts_train, width, label='Train')
    plt.bar(x, counts_val, width, label='Val')  
    plt.bar(x + width, counts_test, width, label='Test')
    plt.xlabel('Breed')
    plt.ylabel('Count')
    plt.title(title)
    plt.xticks(x, breeds, rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.savefig("img/Class_distributions.png")
    plt.show()
    return breed_counts, breed_counts_val, breed_counts_test

traincounts, valcounts, testcounts = plot_class_distribution(train_set, val_set, test_set, 'Train Set Class Distribution')

## Transformations

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
train = TransformSubset(train_set, train_transform)
val = TransformSubset(val_set, val_transform)
test = TransformSubset(test_set, val_transform)

train_labels = [data.samples[idx][1] for idx in train_set.indices]

# Metric Learning Pipeline

In [ ]:
# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# PARAMS
EMBEDDING_DIM = 128
BATCH_SIZE = 64

NUM_EPOCHS = 1


In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        backbone = models.resnet18(pretrained=True)
        self.encoder = nn.Sequential(
            *list(backbone.children())[:-1]
        )
        self.projector = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, embed_dim)
        )

    def forward(self, x):
        feat = self.encoder(x).squeeze(-1).squeeze(-1)
        emb = self.projector(feat)
        return nn.functional.normalize(emb, p=2, dim=1)

In [ ]:
model = EmbeddingNet(embed_dim=EMBEDDING_DIM).cuda()

loss_fn = losses.TripletMarginLoss(margin=0.2)
miner = miners.TripletMarginMiner(
    margin=0.2, type_of_triplets="semihard"
)
sampler = samplers.MPerClassSampler(
    train_labels, m=4, batch_size=BATCH_SIZE
)
train_loader = DataLoader(
    train, batch_size=BATCH_SIZE, sampler=sampler, num_workers=6
)
optimizer = torch.optim.Adam([
    {'params': model.encoder.parameters(), 'lr': 1e-4},
    {'params': model.projector.parameters(), 'lr': 1e-3}
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=30
)

train_losses, val_losses = [], []
for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch+1}")
    model.train()
    epoch_loss = 0
    for images, labels, _ in train_loader:
        embeddings = model(images.cuda())
        hard_pairs = miner(embeddings, labels.cuda())
        loss = loss_fn(embeddings, labels.cuda(), hard_pairs)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    train_losses.append(epoch_loss / len(train_loader))


    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, labels, _ in DataLoader(val, batch_size=BATCH_SIZE):
            embeddings = model(images.cuda())
            hard_pairs = miner(embeddings, labels.cuda())
            loss = loss_fn(embeddings, labels.cuda(), hard_pairs)
            val_loss += loss.item()
    val_losses.append(val_loss / len(DataLoader(val, batch_size=BATCH_SIZE)))
    print(f"Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}")

    if val_losses[-1] < min(val_losses):
        torch.save(model.state_dict(), "models/best_model.pth")


In [ ]:
# plot loss curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Curves')
plt.legend()
plt.savefig('img/loss_curves.png')
plt.show()

# Verification

In [ ]:
test_loader = DataLoader(test, batch_size=64, shuffle=False)
model.eval()
all_embs, all_labels = [], []
with torch.no_grad():
    for imgs, labels, _ in test_loader:
        embs = model(imgs.cuda()).cpu().numpy()
        all_embs.append(embs)
        all_labels.extend(labels.numpy())
embeddings = np.vstack(all_embs)
labels = np.array(all_labels)
np.save('test_embeddings.npy', embeddings)
np.save('test_labels.npy', labels)

In [ ]:
def generate_pairs(labels, n=5000):
    label_to_idx = {}
    for i, l in enumerate(labels):
        label_to_idx.setdefault(l, []).append(i)

    pos = []
    for l, idxs in label_to_idx.items():
        for a, b in itertools.combinations(idxs, 2):
            pos.append((a, b))
    pos = random.sample(pos, min(n, len(pos)))

    neg = []
    all_labs = list(label_to_idx.keys())
    while len(neg) < n:
        l1, l2 = random.sample(all_labs, 2)
        a = random.choice(label_to_idx[l1])
        b = random.choice(label_to_idx[l2])
        neg.append((a, b))
    return pos, neg

In [ ]:
pos_pairs, neg_pairs = generate_pairs(labels)
pair_labels = np.array([1] * len(pos_pairs) + [0] * len(neg_pairs))
similarities = []
for a, b in pos_pairs + neg_pairs:
    sim = np.dot(embeddings[a], embeddings[b])
    similarities.append(sim)

fpr, tpr, thresholds = roc_curve(pair_labels, similarities)
auc = roc_auc_score(pair_labels, similarities)

fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
eer = fpr[eer_idx]
print(f"AUC: {auc:.4f}, EER: {eer:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, linewidth=2, color='#c4622a')
ax.plot([0,1], [0,1], '--', color='#ccc')
ax.scatter([fpr[eer_idx]], [tpr[eer_idx]],
           color='#d44', s=60, zorder=5,
           label=f'EER = {eer:.3f}')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'Verification ROC (AUC = {auc:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig('img/roc_curve.png', dpi=150)

# Retrieval

In [ ]:
# from sklearn.neighbors import NearestNeighbors

# nn = NearestNeighbors(n_neighbors=6, metric='cosine')
# nn.fit(embeddings)
# distances, indices = nn.kneighbors(embeddings)
# # indices[:, 1:] for top-5 excluding self

In [ ]:
# def precision_at_k(query_label, retrieved_labels, k):
#     return np.mean(retrieved_labels[:k] == query_label)

# def recall_at_k(query_label, retrieved_labels, all_labels, k):
#     total_relevant = np.sum(all_labels == query_label) - 1
#     found = np.sum(retrieved_labels[:k] == query_label)
#     return found / max(total_relevant, 1)

# for k in [1, 5]:
#     p = np.mean([precision_at_k(labels[i], labels[indices[i, 1:]], k)
#                  for i in range(len(labels))])
#     r = np.mean([recall_at_k(labels[i], labels[indices[i, 1:]], labels, k)
#                  for i in range(len(labels))])
#     print(f"P@{k}: {p:.4f}  R@{k}: {r:.4f}")

# Few-Shot Classification

In [ ]:
# get embeddings for held-out set
held_out_set_t = TransformSubset(held_out_set, val_transform)
held_out_loader = DataLoader(held_out_set_t, batch_size=64, shuffle=False)
model.eval()
held_out_embs, held_out_labels = [], []
with torch.no_grad():
    for imgs, labels, _ in held_out_loader:
        embs = model(imgs.cuda()).cpu().numpy()
        held_out_embs.append(embs)
        held_out_labels.extend(labels.numpy())


held_out_embeddings = np.vstack(held_out_embs)
held_out_labels = np.array(held_out_labels)

In [ ]:
def run_episodes(embeddings, labels, n_way=5,
                 k_shot=1, n_episodes=100, n_query=15):
    accs = []
    classes = np.unique(labels)
    for _ in range(n_episodes):
        chosen = np.random.choice(classes, n_way, replace=False)
        support_embs, query_embs = [], []
        support_y, query_y = [], []

        for i, c in enumerate(chosen):
            idxs = np.where(labels == c)[0]
            pick = np.random.choice(
                idxs, k_shot + n_query, replace=False
            )
            support_embs.append(embeddings[pick[:k_shot]])
            query_embs.append(embeddings[pick[k_shot:]])
            support_y += [i] * k_shot
            query_y += [i] * n_query

        prototypes = np.array([
            np.mean(s, axis=0) for s in support_embs
        ])
        prototypes /= np.linalg.norm(
            prototypes, axis=1, keepdims=True
        )
        queries = np.vstack(query_embs)
        sims = queries @ prototypes.T
        preds = np.argmax(sims, axis=1)
        accs.append(np.mean(preds == np.array(query_y)))

    return np.mean(accs), np.std(accs)

acc1, std1 = run_episodes(held_out_embs, held_out_labels, k_shot=1)
acc5, std5 = run_episodes(held_out_embs, held_out_labels, k_shot=5)
print(f"1-shot: {acc1:.3f} ± {std1:.3f}")
print(f"5-shot: {acc5:.3f} ± {std5:.3f}")

# Embedding Visualisation